# Visualizing evaluation results

Exploring the parquet files

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from docling_core.types import DoclingDocument
from IPython.display import display
from docling_core.transforms.visualizer.table_visualizer import TableVisualizer
from cells2table.utils.eval import analyze_image, pil_to_cv2, cv2_to_pil

In [ ]:
# Creating the Spark session

spark = SparkSession.builder.appName("cells2table").getOrCreate()

In [ ]:
# Read DoclingDPBench predictions files

benchmark_path = Path("../benchmarks/DoclingDPBench/pred/test/").resolve()

df = spark.read.parquet(str(benchmark_path))

In [ ]:
# Show the schema

df.printSchema(level=1)

In [ ]:
# The ID of the document we want to analyze
doc_id = "doc_bdf0e09acee7a07f7e60c6055128967b36b44564842983ad3b0f273f1c1c5914_page_000001.png"

In [ ]:
row = df.filter(df["document_id"] == doc_id).first()
if row is None:
    raise ValueError("Document not found")

doc = DoclingDocument.model_validate_json(row["PredictedDocument"])

In [ ]:
table = doc.tables[0]
table_img = table.get_image(doc)
if table_img is None:
    raise ValueError("Document generated without generate_page_images=True")
display(table_img)

table_visualizer = TableVisualizer()
pred_img = table_visualizer.get_visualization(doc=doc)[1]
display(pred_img)

In [ ]:
test_img = cv2_to_pil(analyze_image(pil_to_cv2(table_img)))
display(test_img)